<p style ="text-align:center">
    <img src="https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Images/BannerUFPR.png" width="700" alt="PPGOLD/PPGMNE Python:INTRO"  />
</p>

# SAOP09 - Data Science For Business
## Prof. Eduardo Pécora

## Classificação Descritiva com Árvore de Decisão
### Um caso de logística percorrendo a Metodologia de Data Science
Tempo estimado: **90** minutos

### Objetivos

Após completar esta aula, você será capaz de:

* Percorrer as 10 etapas da Metodologia de Data Science em um problema de negócio
* Diferenciar classificação descritiva de classificação preditiva
* Diagnosticar e tratar problemas de qualidade em dados operacionais
* Treinar uma árvore de decisão rasa em um `Pipeline` do scikit-learn
* Traduzir as folhas da árvore em regras de negócio
* Avaliar o modelo pelo desempenho contra um baseline e pela estabilidade das regras

### Roteiro

| # | Etapa | Pergunta que a etapa responde |
|---|---|---|
| 1 | Entendimento do negócio | Qual é o problema e que decisão queremos apoiar? |
| 2 | Abordagem analítica | Que tipo de pergunta é essa e que técnica responde a ela? |
| 3 | Requisitos de dados | Que dados seriam necessários? |
| 4 | Coleta de dados | Que dados temos e de onde vêm? |
| 5 | Entendimento dos dados | O que os dados mostram antes de modelar? |
| 6 | Preparação dos dados | O que precisa ser corrigido e transformado? |
| 7 | Modelagem | Que padrões o modelo encontra? |
| 8 | Avaliação | Os padrões são estáveis e úteis? |
| 9 | Implantação | Como o resultado chega a quem decide? |
| 10 | Feedback | Como saber se continua funcionando? |

> **Sobre os dados:** a base é **sintética**, uma simplificação do processo de distribuição da base `Distribution_12k` (fábricas → CDs → clientes). Ela foi gerada para que os padrões existam de fato e possam ser recuperados pelo modelo, o que permite discutir a metodologia sem que a complexidade dos dados reais domine a aula.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

sns.set_theme(style='whitegrid')
COR = '#2a6f97'          # uma única cor para as barras de taxa de atraso
pd.set_option('display.max_columns', 20)

---
## Etapa 1 — Entendimento do negócio

**Contexto.** Uma distribuidora atende clientes a partir de três centros de distribuição (CD1, CD2, CD3). A última milha (CD → cliente) é feita com três tipos de veículo: *Van*, *MiniVan* e *CityDeliveryTruck*. Cada veículo sai do CD com uma rota de até 6 paradas.

**Problema.** A área comercial recebe reclamações frequentes de atraso. O gerente de operações quer saber **onde** estão os atrasos para agir: trocar o veículo, reordenar as paradas, criar folga em dias de chuva, revisar um CD específico.

**Pergunta de negócio:**
> *Quais combinações de características de uma entrega concentram os atrasos?*

**Decisão que queremos apoiar:** regras de programação de rotas que o planejador consiga aplicar sem software, por exemplo: *"entregas acima de X km não vão de MiniVan"*.

**Critério de sucesso (definido com o gestor, antes de modelar):**
- as regras precisam ser **poucas e legíveis** (cabem em um slide);
- precisam **separar bem** grupos de alto e baixo risco;
- precisam **se repetir** em dados que o modelo não viu.

> Observe que nenhum desses critérios é "maximizar a acurácia". Essa definição vai guiar todas as escolhas seguintes.

---
## Etapa 2 — Abordagem analítica

A pergunta tem uma resposta do tipo **Sim/Não** (a entrega atrasou?), logo é um problema de **classificação binária**.

Mas há duas formas de olhar para um classificador:

| | Classificação **preditiva** | Classificação **descritiva** |
|---|---|---|
| Pergunta | "Esta entrega vai atrasar?" | "Que perfis de entrega atrasam?" |
| Foco | Acertar o rótulo de casos novos | Entender e comunicar o padrão |
| Modelo típico | Ensemble, boosting, redes | Árvore rasa, regressão logística |
| Métrica principal | Acurácia, recall, AUC | Interpretabilidade + estabilidade das regras |

Nosso problema é **descritivo**. A técnica escolhida é a **Árvore de Decisão**, porque:
1. produz regras do tipo **SE ... E ... ENTÃO** que o gestor lê diretamente;
2. captura **interações** (ex.: MiniVan só é problema em distâncias longas) sem que precisemos especificá-las;
3. a profundidade controla diretamente o tamanho da explicação.

> **Para discutir:** se o gestor pedisse "uma lista diária das entregas com maior risco", a abordagem mudaria para preditiva. Mesmo dado, outra pergunta, outra abordagem.

---
## Etapa 3 — Requisitos de dados

**Unidade de análise:** uma entrega (CD → cliente).

**Variável-alvo:** `atraso` (Sim/Não), segundo o prazo prometido ao cliente.

**Hipóteses levantadas com a equipe de operações** — cada uma vira um requisito de dado:

| Hipótese | Variável necessária |
|---|---|
| H1: distâncias longas atrasam mais | distância CD → cliente |
| H2: as últimas paradas da rota acumulam atraso | ordem da entrega na rota |
| H3: o tipo de veículo importa | veículo de última milha |
| H4: chuva atrasa | condição de chuva no dia |
| H5: algum CD opera pior | CD de origem |
| H6: saídas à tarde pegam mais trânsito | janela de saída |
| H7: produto parado muito tempo no estoque atrasa a separação | dias em estoque |
| H8: andar do CD (altura no porta-pallet) atrasa a separação | andar no CD |
| H9: cargas pesadas atrasam | peso da carga |

> Nem toda hipótese será confirmada. Uma das funções do modelo descritivo é justamente **descartar** hipóteses.

---
## Etapa 4 — Coleta de dados

Os dados vêm de duas fontes integradas pela equipe de TI: o **sistema de roteirização** (CD, veículo, distância, ordem, janela, peso), o **WMS** do armazém (dias em estoque, andar) e uma base meteorológica (chuva). O rótulo `atraso` vem do **comprovante de entrega**.

### Dicionário de dados

| Coluna | Tipo | Descrição |
|---|---|---|
| `id_entrega` | texto | Identificador da entrega |
| `cd` | categórica | Centro de distribuição de origem (CD1, CD2, CD3) |
| `veiculo` | categórica | Veículo de última milha (Van, MiniVan, CityDeliveryTruck) |
| `distancia_km` | numérica | Distância CD → cliente (km) |
| `ordem_rota` | numérica | Posição da entrega na rota (1 a 6) |
| `janela` | categórica | Janela de saída do veículo (Manhã, Tarde) |
| `chuva` | categórica | Choveu no dia da entrega? (Sim, Não) |
| `dias_estoque` | numérica | Dias que o produto ficou estocado no CD |
| `andar_cd` | numérica | Nível do porta-pallet onde o produto estava (0 a 4) |
| `peso_kg` | numérica | Peso da carga da entrega (kg) |
| `atraso` | categórica | **Alvo**: a entrega atrasou? (Sim, Não) |

In [ ]:
URL = 'https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Dados/Entregas_Atraso.csv'
df_raw = pd.read_csv(URL)
print(df_raw.shape)
df_raw.head()

---
## Etapa 5 — Entendimento dos dados

Antes de modelar, olhamos os dados por dois ângulos:
1. **Qualidade** — tipos, faltantes, valores impossíveis, categorias inconsistentes, duplicatas;
2. **Conteúdo** — distribuição do alvo e relação de cada variável com o atraso.

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe().T

**O que procurar no `describe()`:** valores mínimos e máximos de `distancia_km` fazem sentido para entregas urbanas e regionais?

In [ ]:
# Categorias de cada variável texto
for c in df_raw.select_dtypes(exclude='number').columns.drop('id_entrega'):
    print(f'{c}: {df_raw[c].value_counts(dropna=False).to_dict()}')

In [ ]:
print('Faltantes por coluna:')
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
print('\nLinhas duplicadas:', df_raw.duplicated().sum())
print('Distâncias impossíveis (< 0 ou > 300 km):',
      ((df_raw['distancia_km'] < 0) | (df_raw['distancia_km'] > 300)).sum())

### Diagnóstico de qualidade
Encontramos quatro problemas típicos de dados operacionais:
- **categorias escritas de formas diferentes** em `veiculo` (`van`, `Van `, `minivan`) e em `chuva` (`sim`);
- **faltantes** em `distancia_km`;
- **valores impossíveis** de distância (negativos e milhares de km — provável erro de digitação);
- **linhas duplicadas** (mesma entrega registrada duas vezes).

Eles serão tratados na Etapa 6. Para a análise exploratória do conteúdo, fazemos uma limpeza rápida apenas das categorias.

In [ ]:
# Distribuição do alvo
taxa_geral = (df_raw['atraso'] == 'Sim').mean()
print(f'Taxa geral de atraso: {taxa_geral:.1%}')

ax = df_raw['atraso'].value_counts().plot.bar(color=COR, width=0.5, rot=0)
ax.set_title('Entregas por situação')
ax.set_xlabel('')
ax.set_ylabel('Número de entregas')
plt.show()

### Taxa de atraso por variável
Para um problema de classificação descritivo, o gráfico mais útil não é a contagem, e sim a **taxa de atraso dentro de cada grupo**. A linha tracejada é a taxa geral: barras acima dela indicam grupos com mais atraso que a média.

In [ ]:
eda = df_raw.copy()
eda['veiculo'] = eda['veiculo'].str.strip().str.lower().map(
    {'van': 'Van', 'minivan': 'MiniVan', 'citydeliverytruck': 'CityDeliveryTruck'})
eda['chuva'] = eda['chuva'].str.capitalize()
eda['atrasou'] = (eda['atraso'] == 'Sim').astype(int)
eda['faixa_dist'] = pd.cut(eda['distancia_km'], bins=[0, 60, 90, 130, 160, 200])

vars_plot = ['cd', 'veiculo', 'janela', 'chuva', 'ordem_rota', 'faixa_dist',
             'andar_cd', 'faixa_estoque', 'faixa_peso']
eda['faixa_estoque'] = pd.cut(eda['dias_estoque'], bins=[0, 10, 20, 30, 40])
eda['faixa_peso'] = pd.qcut(eda['peso_kg'], q=4)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, v in zip(axes.ravel(), vars_plot):
    taxa = eda.groupby(v, observed=True)['atrasou'].mean()
    taxa.plot.bar(ax=ax, color=COR, width=0.6, rot=30)
    ax.axhline(taxa_geral, ls='--', color='gray', lw=1)
    ax.set_title(v)
    ax.set_xlabel('')
    ax.set_ylim(0, 0.8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0, 0].set_ylabel('Taxa de atraso')
fig.suptitle('Taxa de atraso por grupo (linha tracejada = taxa geral)', y=1.0)
plt.tight_layout()
plt.show()

### Leitura da exploração
- **Distância** e **ordem na rota** mostram os maiores contrastes.
- **Chuva** e **MiniVan** também elevam a taxa.
- **CD** e **janela** mostram diferenças pequenas.
- **Andar no CD**, **dias em estoque** e **peso** não parecem ter relação com o atraso.

Essa análise é **univariada**: olha uma variável de cada vez. Ela não responde, por exemplo, se a MiniVan é ruim **sempre** ou **apenas em distâncias longas**. Para combinações, precisamos do modelo.

In [ ]:
# Uma prévia de interação: veículo x faixa de distância
pivo = eda.pivot_table(index='veiculo', columns='faixa_dist', values='atrasou',
                       aggfunc='mean', observed=True)
plt.figure(figsize=(9, 3.5))
sns.heatmap(pivo, annot=True, fmt='.0%', cmap='Blues', cbar=False)
plt.title('Taxa de atraso: veículo x faixa de distância')
plt.ylabel('')
plt.xlabel('Distância (km)')
plt.show()

O mapa de calor sugere que a MiniVan só se destaca a partir de certa distância. Esse tipo de **interação** é exatamente o que uma árvore de decisão encontra sozinha.

---
## Etapa 6 — Preparação dos dados

Vamos tratar cada problema encontrado na Etapa 5, sempre registrando **quantas linhas** cada decisão afeta.

In [ ]:
df = df_raw.copy()
n0 = len(df)

# 1) Duplicatas
df = df.drop_duplicates()
print(f'Duplicatas removidas: {n0 - len(df)}')

# 2) Padronização de categorias
mapa_veiculo = {'van': 'Van', 'minivan': 'MiniVan', 'citydeliverytruck': 'CityDeliveryTruck'}
df['veiculo'] = df['veiculo'].str.strip().str.lower().map(mapa_veiculo)
df['chuva'] = df['chuva'].str.strip().str.capitalize()
print('veiculo:', df['veiculo'].value_counts().to_dict())
print('chuva  :', df['chuva'].value_counts().to_dict())

# 3) Valores impossíveis: removidos (não há como saber o valor correto)
impossivel = (df['distancia_km'] < 0) | (df['distancia_km'] > 300)
print(f'Distâncias impossíveis removidas: {impossivel.sum()}')
df = df[~impossivel]

print(f'\nLinhas: {n0} -> {len(df)}')
print(f'Faltantes restantes em distancia_km: {df["distancia_km"].isna().sum()}')

### E os faltantes de distância?
Os faltantes **não** são preenchidos aqui. A imputação pela mediana será feita **dentro do pipeline**, depois da separação treino/teste, para que a mediana seja calculada apenas com os dados de treino.

> Detalhe da regra de valores impossíveis: comparações com `NaN` (`NaN < 0`, `NaN > 300`) retornam `False`, então as linhas com distância faltante **não** foram removidas — que é o que queremos.

### Alvo, preditoras e separação treino/teste
Em um modelo **descritivo**, para que serve o conjunto de teste? Para verificar se as regras encontradas **se repetem** em dados que o modelo não viu, isto é, se descrevem o processo e não as particularidades da amostra.

In [ ]:
X = df.drop(columns=['id_entrega', 'atraso'])
y = (df['atraso'] == 'Sim').astype(int)        # 1 = atrasou

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f'Treino: {X_train.shape}  | taxa de atraso = {y_train.mean():.1%}')
print(f'Teste : {X_test.shape}  | taxa de atraso = {y_test.mean():.1%}')

In [ ]:
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()
print('Categóricas:', cat_cols)
print('Numéricas  :', num_cols)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), cat_cols),
    ('num', SimpleImputer(strategy='median'), num_cols),
], verbose_feature_names_out=False)

**Categorias de referência** (removidas pelo `drop='first'`, em ordem alfabética):

| Variável | Referência | Colunas criadas |
|---|---|---|
| `cd` | CD1 | `cd_CD2`, `cd_CD3` |
| `veiculo` | CityDeliveryTruck | `veiculo_MiniVan`, `veiculo_Van` |
| `janela` | Manhã | `janela_Tarde` |
| `chuva` | Não | `chuva_Sim` |

Na árvore, uma divisão como `chuva_Sim <= 0.5` significa simplesmente **"não choveu"**.

---
## Etapa 7 — Modelagem

### Primeiro: o que acontece sem limitar a árvore?

In [ ]:
arvore_livre = Pipeline([('prep', preprocess),
                         ('tree', DecisionTreeClassifier(random_state=42))])
arvore_livre.fit(X_train, y_train)

t = arvore_livre.named_steps['tree']
print(f'Profundidade: {t.get_depth()}  |  Folhas (regras): {t.get_n_leaves()}')
print(f'Acurácia treino: {arvore_livre.score(X_train, y_train):.3f}')
print(f'Acurácia teste : {arvore_livre.score(X_test, y_test):.3f}')

Centenas de regras, acurácia perfeita no treino e queda forte no teste. A árvore **decorou** a amostra. Além de não generalizar, ela não cumpre nenhum dos critérios de sucesso da Etapa 1: ninguém consegue ler centenas de regras.

### Árvore descritiva: rasa e com folhas grandes
- `max_depth=3` → no máximo 3 condições por regra e até 8 regras;
- `min_samples_leaf=40` → cada regra precisa valer para pelo menos 40 entregas, evitando regras baseadas em poucos casos.

In [ ]:
modelo = Pipeline([
    ('prep', preprocess),
    ('tree', DecisionTreeClassifier(max_depth=3, min_samples_leaf=40, random_state=42)),
])
modelo.fit(X_train, y_train)

arvore = modelo.named_steps['tree']
nomes = modelo.named_steps['prep'].get_feature_names_out()
print(f'Folhas (regras): {arvore.get_n_leaves()}')

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(arvore, feature_names=nomes, class_names=['No prazo', 'Atraso'],
          filled=True, rounded=True, proportion=True, impurity=False, fontsize=10)
plt.title('Árvore de decisão descritiva (profundidade 3)')
plt.show()

**Como ler cada nó:** a primeira linha é a condição; `samples` é a fração das entregas de treino que chega ao nó; `value` são as proporções [no prazo, atraso]; a cor fica mais intensa quanto mais "puro" o nó.

In [ ]:
print(export_text(arvore, feature_names=list(nomes), show_weights=False))

### Traduzindo a árvore em regras de negócio
A função abaixo percorre a árvore e escreve cada folha como uma regra **SE ... ENTÃO**, traduzindo as colunas one-hot de volta para a linguagem do negócio.

In [ ]:
def traduz(nome, sinal, limiar):
    # Converte uma condição da árvore em texto legível
    for c in cat_cols:
        if nome.startswith(c + '_'):
            cat = nome[len(c) + 1:]
            return f'{c} = {cat}' if sinal == '>' else f'{c} ≠ {cat}'
    return f'{nome} {sinal} {limiar:.1f}'

def extrai_regras(tree, nomes):
    tr = tree.tree_
    regras = {}
    def visita(no, conds):
        if tr.children_left[no] == -1:            # é folha
            regras[no] = ' E '.join(conds)
            return
        f, lim = nomes[tr.feature[no]], tr.threshold[no]
        visita(tr.children_left[no],  conds + [traduz(f, '<=', lim)])
        visita(tr.children_right[no], conds + [traduz(f, '>',  lim)])
    visita(0, [])
    return regras

regras = extrai_regras(arvore, nomes)

folha_tr = arvore.apply(modelo.named_steps['prep'].transform(X_train))
tab = (pd.DataFrame({'folha': folha_tr, 'atraso': y_train.values})
         .groupby('folha')['atraso'].agg(n_treino='size', taxa_treino='mean'))
tab['regra'] = tab.index.map(regras)
tab['% das entregas'] = tab['n_treino'] / tab['n_treino'].sum()
tab = tab.sort_values('taxa_treino', ascending=False)

pd.set_option('display.max_colwidth', None)
(tab[['regra', 'n_treino', '% das entregas', 'taxa_treino']]
   .style.format({'% das entregas': '{:.1%}', 'taxa_treino': '{:.1%}'})
   .bar(subset='taxa_treino', color='#9ecae1', vmin=0, vmax=1))

In [ ]:
imp = pd.Series(arvore.feature_importances_, index=nomes).sort_values()
ax = imp.plot.barh(color=COR, figsize=(8, 5))
ax.set_title('Importância das variáveis na árvore')
ax.set_xlabel('Importância (redução de impureza)')
plt.show()

### Confrontando com as hipóteses da Etapa 3

| Hipótese | Resultado |
|---|---|
| H1 distância | **Confirmada** — é a primeira divisão (~130 km) |
| H2 ordem na rota | **Confirmada** — paradas 5 e 6 atrasam mais |
| H3 veículo | **Confirmada com interação** — MiniVan só é problema em distância longa |
| H4 chuva | **Confirmada** — agrava as últimas paradas em distâncias curtas |
| H5 CD, H6 janela | Efeito pequeno: não entraram em uma árvore de profundidade 3 |
| H7, H8, H9 estoque, andar, peso | **Não encontradas** — importância zero |

> "Não entrou na árvore" não é o mesmo que "não tem efeito nenhum". Significa que, com apenas 3 níveis, há variáveis que explicam mais. Isso é uma escolha de **simplicidade**, coerente com o objetivo descritivo.

---
## Etapa 8 — Avaliação

Avaliamos o modelo descritivo em três frentes, retomando os critérios da Etapa 1:
1. **Desempenho**: o modelo é melhor do que não ter modelo?
2. **Estabilidade**: as regras se repetem no conjunto de teste?
3. **Simplicidade**: a profundidade escolhida é razoável?

### 8.1 Desempenho contra uma referência (baseline)

In [ ]:
baseline = 1 - y_test.mean()      # prever sempre "no prazo"
y_pred = modelo.predict(X_test)

print(f'Baseline (sempre "no prazo") : {baseline:.3f}')
print(f'Árvore - acurácia treino     : {modelo.score(X_train, y_train):.3f}')
print(f'Árvore - acurácia teste      : {accuracy_score(y_test, y_pred):.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['No prazo', 'Atraso']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de confusão (teste)')
plt.show()

print(classification_report(y_test, y_pred, target_names=['No prazo', 'Atraso']))

A árvore supera o baseline, e treino e teste ficam próximos (não há sobreajuste). O recall de atraso não é alto: muitas entregas atrasam fora dos grupos de risco, porque o atraso tem também uma parte **aleatória** (trânsito, cliente ausente...).

> Para um modelo **descritivo** isso não é um defeito. O objetivo não é acertar cada entrega, e sim mostrar onde o risco **se concentra**.

### 8.2 Estabilidade das regras
Aplicamos as mesmas regras ao conjunto de teste e comparamos a taxa de atraso de cada grupo.

In [ ]:
folha_te = arvore.apply(modelo.named_steps['prep'].transform(X_test))
te = (pd.DataFrame({'folha': folha_te, 'atraso': y_test.values})
        .groupby('folha')['atraso'].agg(n_teste='size', taxa_teste='mean'))
estab = tab.join(te)[['regra', 'n_treino', 'taxa_treino', 'n_teste', 'taxa_teste']]
estab.style.format({'taxa_treino': '{:.1%}', 'taxa_teste': '{:.1%}'})

In [ ]:
ordem = estab.sort_values('taxa_treino')
pos = np.arange(len(ordem))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(pos - 0.2, ordem['taxa_treino'], height=0.38, color=COR, label='Treino')
ax.barh(pos + 0.2, ordem['taxa_teste'], height=0.38, color='#9ecae1', label='Teste')
ax.set_yticks(pos, [f'Folha {i}' for i in ordem.index])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_xlabel('Taxa de atraso')
ax.set_title('Taxa de atraso por regra: treino x teste')
ax.legend(loc='lower right')
plt.show()

A ordem dos grupos se mantém no teste: as regras de alto risco continuam sendo de alto risco. **As regras são estáveis.**

### 8.3 Escolha da profundidade

In [ ]:
res = []
for d in range(1, 11):
    m = Pipeline([('prep', preprocess),
                  ('tree', DecisionTreeClassifier(max_depth=d, min_samples_leaf=40, random_state=42))])
    m.fit(X_train, y_train)
    res.append({'profundidade': d,
                'regras': m.named_steps['tree'].get_n_leaves(),
                'treino': m.score(X_train, y_train),
                'teste': m.score(X_test, y_test)})
res = pd.DataFrame(res)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(res['profundidade'], res['treino'], marker='o', lw=2, label='Treino')
ax.plot(res['profundidade'], res['teste'], marker='o', lw=2, label='Teste')
ax.axvline(3, ls='--', color='gray', lw=1)
ax.text(0.27, 0.05, 'profundidade escolhida', transform=ax.transAxes, color='gray')
ax.set_xlabel('Profundidade máxima')
ax.set_ylabel('Acurácia')
ax.set_title('Acurácia x profundidade da árvore')
ax.legend()
plt.show()

res.round(3)

A partir da profundidade 3, a acurácia de teste praticamente não melhora, enquanto o número de regras cresce rapidamente. **Profundidade 3 é o melhor equilíbrio entre explicação e desempenho** — exatamente o que pede um modelo descritivo.

---
## Etapa 9 — Implantação

Em um modelo descritivo, a "implantação" geralmente **não é um sistema**: é a incorporação das regras ao processo de decisão. Três entregáveis possíveis:

**1. Quadro de regras para o planejador de rotas** (a tabela da Etapa 7, ordenada por risco).

**2. Recomendações operacionais**, derivadas diretamente das regras:
- não programar **MiniVan** para entregas acima de ~130 km;
- em rotas longas, colocar os clientes com prazo mais crítico **nas primeiras paradas**;
- em dias de **chuva**, rever o prazo prometido das paradas 5 e 6.

**3. Classificação de risco no momento da programação**, usando o próprio modelo:

In [ ]:
novas = pd.DataFrame({
    'cd':           ['CD1',  'CD3',     'CD2',  'CD1'],
    'veiculo':      ['Van',  'MiniVan', 'Van',  'CityDeliveryTruck'],
    'distancia_km': [45.0,   150.0,     110.0,  170.0],
    'ordem_rota':   [2,      3,         6,      5],
    'janela':       ['Manhã','Tarde',   'Manhã','Tarde'],
    'chuva':        ['Não',  'Não',     'Sim',  'Não'],
    'dias_estoque': [12,     30,        5,      20],
    'andar_cd':     [0,      3,         1,      2],
    'peso_kg':      [250,    400,       150,    600],
})

novas['prob_atraso'] = modelo.predict_proba(novas)[:, 1]
novas['risco'] = pd.cut(novas['prob_atraso'], bins=[0, 0.3, 0.6, 1],
                        labels=['Baixo', 'Médio', 'Alto'])
novas['regra'] = arvore.apply(modelo.named_steps['prep'].transform(novas)).tolist()
novas['regra'] = novas['regra'].map(regras)
novas[['veiculo', 'distancia_km', 'ordem_rota', 'chuva', 'prob_atraso', 'risco', 'regra']]

Note que a probabilidade de cada nova entrega é simplesmente a **taxa de atraso da regra** em que ela cai. Em uma árvore, a previsão e a explicação são a mesma coisa.

---
## Etapa 10 — Feedback

A metodologia é um **ciclo**. Depois de implantar as recomendações:
- **Monitorar** a taxa de atraso mensal, por regra. Se a regra "MiniVan + distância longa" sumir porque a MiniVan deixou de fazer essas rotas, isso é sucesso, não falha do modelo.
- **Re-treinar** periodicamente: novas rotas, novos veículos ou um novo CD mudam o padrão.
- **Voltar à Etapa 1** se surgirem novas perguntas — por exemplo, "quanto custa cada atraso?" leva a outra variável-alvo e outra abordagem.
- **Voltar à Etapa 3** se o time sugerir novas hipóteses (motorista, trânsito em tempo real, tipo de cliente) — novas variáveis exigem nova coleta.

---
## Exercícios

1. **(Etapa 7)** Treine a árvore com `max_depth=2`. Quais regras sobrevivem? O que se perde em relação à profundidade 3?
2. **(Etapa 7)** Treine com `max_depth=4`. As variáveis `cd` ou `janela` aparecem? A explicação melhora o suficiente para justificar as regras adicionais?
3. **(Etapa 6)** Em vez de remover as distâncias impossíveis, transforme-as em `NaN` e deixe o pipeline imputar a mediana. As regras mudam?
4. **(Etapa 8)** Use `class_weight='balanced'` na árvore. O que acontece com o recall de atraso e com as regras? Em que situação de negócio essa troca valeria a pena?
5. **(Etapa 2)** Escreva, em um parágrafo, como seria esta mesma análise se o gestor pedisse uma **lista diária das entregas de maior risco**. O que muda nas Etapas 2, 8 e 9?

## Fique Conectado

- [![YouTube](https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Images/youtube_icon_color.png)](https://www.youtube.com/@LigaDataScience/videos)  
  Explore nossos vídeos educacionais e webinars sobre ciência de dados, machine learning e inteligência artificial. Inscreva-se para não perder nenhuma atualização!

- [![LinkedIn](https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Images/linkedin_icon_color.png)](https://www.linkedin.com/company/liga-data-science-ufpr/)  
  Siga-nos no LinkedIn para as últimas novidades, oportunidades de carreira e networking profissional no campo da ciência de dados.

- [![Instagram](https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Images/instagram_icon_color.png)](https://www.instagram.com/ligadatascience/)  
  Confira nosso Instagram para conteúdos dos bastidores, destaques de eventos e o dia a dia da Liga Data Science. Faça parte da nossa jornada!


## Autores

<a href="https://www.linkedin.com/in/eduardopecora/" target="_blank">Eduardo Pecora</a>

## Referências:

Metodologia de Data Science baseada no material da IBM (John Rollins, *Foundational Methodology for Data Science*).
<br>

* Documentação da biblioteca <a href="https://scikit-learn.org/stable/modules/tree.html">scikit-learn — Árvores de decisão</a>
* Documentação da biblioteca <a href="https://pandas.pydata.org/docs/">Pandas</a>
* Documentação da biblioteca <a href="https://seaborn.pydata.org/">Seaborn</a>
* Documentação da biblioteca <a href="https://matplotlib.org/">Matplotlib</a>

## Log de modificações

| Data | Versão | Modificado por | Descrição |
| -----------| ------- | ---------- | ---------------------------------- |
| 13-11-2026       | 1.0     | Eduardo Pecora    | Inicial               |